In [ ]:
# Cell 1: Unpack verified project codebase, install GPU pose runtime, and run 12/12 unit tests
import base64, io, os, subprocess, sys, tarfile
from pathlib import Path

PROJ_DIR = Path('/kaggle/working/shortpose_proj') if Path('/kaggle/working').exists() else Path('/content/shortpose_proj')
PROJ_DIR.mkdir(parents=True, exist_ok=True)

PAYLOAD_B64 = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD_B64)), mode='r:gz') as tf:
    tf.extractall(PROJ_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx2torch', 'onnx', 'onnxruntime-gpu', 'gdown'], check=True)
sys.path.insert(0, str(PROJ_DIR))
os.environ['PYTHONPATH'] = str(PROJ_DIR)

import torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available(), 'GPU count:', torch.cuda.device_count(), flush=True)
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i), flush=True)

subprocess.run([sys.executable, str(PROJ_DIR / 'tests/test_invariants.py')], check=True)


In [ ]:
# Cell 2: Stage RTMPose-m ONNX, PINNS (10Hz real video + homography), and JTA (2.5Hz) in /tmp
from pathlib import Path
from scripts.cloud_experiment_runner import stage_and_extract_all_datasets

PINNS_SAMPLES, JTA_SAMPLES, DATASET_DIAG = stage_and_extract_all_datasets(Path('/tmp/shortpose_raw'))
print('Dataset Diagnostics:', DATASET_DIAG, flush=True)


In [ ]:
# Cell 3: Execute 5-seed controlled experiments (Suites A-E), Holm-Bonferroni stats, and export bundle
import shutil
from pathlib import Path
from scripts.cloud_experiment_runner import execute_suites_from_samples

OUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content/output')
BUNDLE_PATH = execute_suites_from_samples(
    PINNS_SAMPLES, JTA_SAMPLES, DATASET_DIAG, out_dir=OUT_ROOT, epochs_pinns=28, epochs_jta=24
)
rep_txt = (OUT_ROOT / 'results' / 'master_experiment_report.json').read_text(encoding='utf-8')
print('=== MASTER_REPORT_BYTES ===', len(rep_txt))
if (OUT_ROOT / 'shortpose_proj').exists():
    shutil.rmtree(OUT_ROOT / 'shortpose_proj', ignore_errors=True)
shutil.rmtree('/tmp/shortpose_raw', ignore_errors=True)
print('READY FOR DOWNLOAD:', BUNDLE_PATH)
try:
    from google.colab import files
    files.download(str(BUNDLE_PATH))
    files.download(str(OUT_ROOT / 'results' / 'master_experiment_report.json'))
except Exception as exc:
    print('Colab auto-download skipped:', exc)
